# Dataset v2: подбор веса SSIM для shared-depth

Сравниваются четыре строго сопоставимых запуска:

- `BaselineUNet`, вес SSIM `1.1`;
- `SharedDepthUNet`, веса SSIM `1.1`, `1.5`, `2.0`.

Dataset, subset, инициализация соответствующих shared-depth моделей, порядок батчей и физика совпадают.
Валидация дополнительно разделяет procedural и BSDS. Это короткий screen, не финальное обучение.

In [ ]:
from __future__ import annotations

import csv
import gc
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

try:
    from pytorch_msssim import ssim
except ImportError:
    print('Installing pytorch-msssim...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v2')
OUTPUT_DIR = Path('/kaggle/working/ssim_weight_screen_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 128
WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)
DATA_SEED = 42
TRAIN_SEED = 42
TRAIN_SAMPLES = 512
VAL_SAMPLES = 128
EPOCHS = 6
BATCH_SIZE = 16
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(2, os.cpu_count() or 1)
LOG_EVERY = 10

def find_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_root(KAGGLE_DATASET_DIR)
metadata = json.loads((DATA_ROOT / 'dataset_metadata.json').read_text())
if metadata.get('schema_version') != 2:
    raise RuntimeError(f'Expected dataset schema 2, got {metadata.get("schema_version")}')
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'
print('DATA_ROOT:', DATA_ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'disabled')

In [ ]:
RESAMPLE = Image.Resampling.LANCZOS

def normalize_image(array: np.ndarray) -> torch.Tensor:
    values = torch.from_numpy(np.asarray(array, dtype=np.float32) / 255.0)
    maximum = values.amax()
    if maximum <= 1e-8:
        raise ValueError('All-black image cannot be normalized')
    return values / maximum

def font_for_size(size: int):
    try:
        return ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError:
        return ImageFont.load_default()

def draw_primitive(mask: Image.Image, rng: np.random.Generator, remaining_fraction: float) -> None:
    draw = ImageDraw.Draw(mask)
    target_area = max(12.0, remaining_fraction * IMAGE_SIZE * IMAGE_SIZE * float(rng.uniform(0.25, 0.65)))
    kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'glyph'], p=[0.28, 0.28, 0.16, 0.14, 0.14]))
    aspect = float(np.exp(rng.uniform(np.log(0.45), np.log(2.2))))
    width = int(np.clip(np.sqrt(target_area * aspect), 3, IMAGE_SIZE - 4))
    height = int(np.clip(np.sqrt(target_area / aspect), 3, IMAGE_SIZE - 4))
    x0 = int(rng.integers(2, max(3, IMAGE_SIZE - width - 1)))
    y0 = int(rng.integers(2, max(3, IMAGE_SIZE - height - 1)))
    x1, y1 = min(IMAGE_SIZE - 2, x0 + width), min(IMAGE_SIZE - 2, y0 + height)

    if kind == 'ellipse':
        draw.ellipse((x0, y0, x1, y1), fill=255)
    elif kind == 'rectangle':
        draw.rectangle((x0, y0, x1, y1), fill=255)
    elif kind == 'line':
        line_width = max(1, int(np.clip(target_area / max(width, height), 1, 10)))
        draw.line((x0, y0, x1, y1), fill=255, width=line_width)
    elif kind == 'ring':
        ring_width = max(1, int(np.clip(min(width, height) * 0.12, 1, 9)))
        draw.ellipse((x0, y0, x1, y1), outline=255, width=ring_width)
    else:
        char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
        font = font_for_size(max(10, min(80, height)))
        glyph = Image.new('L', (max(16, width * 2), max(16, height * 2)), 0)
        gd = ImageDraw.Draw(glyph)
        box = gd.textbbox((0, 0), char, font=font)
        gd.text((-box[0], -box[1]), char, fill=255, font=font)
        box = glyph.getbbox()
        if box is not None:
            glyph = glyph.crop(box).resize((max(3, width), max(3, height)), RESAMPLE)
            mask.paste(glyph, (x0, y0), glyph)

def make_mask(seed: int, target_coverage: float) -> tuple[Image.Image, float]:
    best_mask, best_coverage, best_error = None, 0.0, float('inf')
    for attempt in range(8):
        rng = np.random.default_rng(seed + 1009 * attempt)
        mask = Image.new('L', (IMAGE_SIZE, IMAGE_SIZE), 0)
        for _ in range(18):
            coverage = float((np.asarray(mask) > 16).mean())
            error = abs(coverage - target_coverage)
            if error < best_error:
                best_mask, best_coverage, best_error = mask.copy(), coverage, error
            if coverage >= target_coverage * 0.96:
                break
            draw_primitive(mask, rng, max(target_coverage - coverage, 0.002))
        coverage = float((np.asarray(mask) > 16).mean())
        error = abs(coverage - target_coverage)
        if error < best_error:
            best_mask, best_coverage, best_error = mask.copy(), coverage, error
        if target_coverage * 0.75 <= coverage <= target_coverage * 1.25:
            return mask, coverage
    return best_mask, best_coverage

def render_synthetic(item: dict, return_coverage: bool = False):
    seed = int(item['seed'])
    rng = np.random.default_rng(seed)
    target = float(item['target_coverage'])
    mask, coverage = make_mask(seed, target)
    mode = item['background_mode']
    if mode == 'dark':
        background = int(rng.integers(0, 19)); foreground = int(rng.integers(max(80, background + 65), 256))
    elif mode == 'light':
        background = int(rng.integers(175, 246)); foreground = int(rng.integers(0, max(1, background - 65)))
    else:
        background = int(rng.integers(25, 101))
        if rng.random() < 0.7:
            foreground = int(rng.integers(min(255, background + 65), 256))
        else:
            foreground = int(rng.integers(0, max(1, background - 20)))
    mask_values = np.asarray(mask, dtype=np.float32) / 255.0
    image = background + mask_values * (foreground - background)
    image = np.clip(np.rint(image), 0, 255).astype(np.uint8)
    tensor = normalize_image(image)
    return (tensor, coverage) if return_coverage else tensor

def load_natural(item: dict, root: Path = BSDS_ROOT) -> torch.Tensor:
    path = root / item['path']
    with Image.open(path) as source:
        image = source.convert('L')
        if item['transform'] == 'full':
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), method=RESAMPLE, color=fill, centering=(0.5, 0.5))
        else:
            crop = item['crop']; size = int(crop['size']); x, y = int(crop['x']), int(crop['y'])
            image = image.crop((x, y, x + size, y + size)).resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE)
        if item.get('hflip', False):
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        angle = int(item.get('rotation', 0))
        if angle:
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = image.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=fill)
        return normalize_image(np.asarray(image, dtype=np.uint8))

def load_item(item: dict) -> torch.Tensor:
    return render_synthetic(item) if item['source'] == 'synthetic' else load_natural(item)

class QuadDataset(Dataset):
    def __init__(self, manifest: Path):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        quad = torch.stack([load_item(item) for item in self.records[index]['items']]).to(torch.float32)
        if quad.shape != (4, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f'Invalid quad shape: {tuple(quad.shape)}')
        return quad

class QuadSourceDataset(QuadDataset):
    def __getitem__(self, index):
        record = self.records[index]
        quad = torch.stack([load_item(item) for item in record['items']]).to(torch.float32)
        sources = torch.tensor([1 if item['source'] == 'bsds' else 0 for item in record['items']], dtype=torch.long)
        return quad, sources

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())

class UpBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)
    def forward(self, x, skip):
        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.block(torch.cat((self.reduce(x), skip), dim=1))

class BaselineUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e0, self.e1, self.e2, self.e3 = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)
        self.pool, self.b = nn.MaxPool2d(2), DoubleConv(256, 512)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.b(self.pool(e3)); x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class DepthEmbedding(nn.Module):
    def __init__(self, channels):
        super().__init__(); hidden = max(16, channels // 2)
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, channels))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, depths):
        return self.net(depths[:, None])[:, :, None, None]

class PlaneFusion(nn.Sequential):
    def __init__(self, channels):
        super().__init__(nn.Conv2d(4 * channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU())

class SharedDepthUNet(nn.Module):
    def __init__(self):
        super().__init__(); self.planes = 4; channels = (32, 64, 128, 256, 512)
        self.encoders = nn.ModuleList([DoubleConv(1, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256), DoubleConv(256, 512)])
        self.depth_embeddings = nn.ModuleList([DepthEmbedding(c) for c in channels])
        self.fusions = nn.ModuleList([PlaneFusion(c) for c in channels])
        self.pool = nn.MaxPool2d(2)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
        z = torch.tensor(ZS, dtype=torch.float32) * 1e3
        self.register_buffer('depths', (z - z.mean()) / (z.max() - z.min()))
    def forward(self, x):
        batch, planes, height, width = x.shape
        features = x.reshape(batch * planes, 1, height, width); depths = self.depths.repeat(batch); fused = []
        for level, (encoder, embedding, fusion) in enumerate(zip(self.encoders, self.depth_embeddings, self.fusions)):
            features = encoder(features) + embedding(depths).to(features.dtype)
            _, channels, h, w = features.shape
            fused.append(fusion(features.reshape(batch, planes * channels, h, w)))
            if level + 1 < len(self.encoders): features = self.pool(features)
        e0, e1, e2, e3, x = fused
        x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class FresnelLayer(nn.Module):
    def __init__(self, device):
        super().__init__(); c = (torch.arange(IMAGE_SIZE, dtype=torch.float64) - IMAGE_SIZE / 2) * PIXEL_PITCH
        u, v = torch.meshgrid(c, c, indexing='ij')
        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]
        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)
    def forward(self, phase):
        field = torch.polar(torch.ones_like(phase, dtype=torch.float32), phase.float()); outputs = []
        for kernel in self.kernels:
            value = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()
            outputs.append(value)
        amplitude = torch.cat(outputs, dim=1)
        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)

def pearson_cc(a, b, eps=1e-8):
    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)
    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)

def doe_loss(target, reconstruction, ssim_weight):
    cc = pearson_cc(target, reconstruction)
    sv = torch.stack([ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False) for m in range(4)], dim=1)
    return (1 - cc).mean() + ssim_weight * (1 - sv).mean(), cc, sv

## Обучение и выбор веса

Кандидат допускается к полному обучению, если относительно baseline:

- `ΔCC ≥ +0.01`;
- `ΔSSIM ≥ −0.003`;
- `Δ worst CC ≥ 0`;
- `Δ worst SSIM ≥ −0.003`.

Среди прошедших кандидатов выбирается вариант с наибольшим SSIM, затем с наибольшим CC.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle GPU: Settings -> Accelerator -> GPU')
device = torch.device('cuda'); physics = FresnelLayer(device)

full_train = QuadSourceDataset(DATA_ROOT / 'manifests' / 'train.jsonl')
full_val = QuadSourceDataset(DATA_ROOT / 'manifests' / 'val.jsonl')
subset_rng = np.random.default_rng(DATA_SEED + 101)
train_indices = subset_rng.choice(len(full_train), TRAIN_SAMPLES, replace=False).tolist()
val_indices = subset_rng.choice(len(full_val), VAL_SAMPLES, replace=False).tolist()
train_set, val_set = Subset(full_train, train_indices), Subset(full_val, val_indices)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def make_loaders():
    generator = torch.Generator().manual_seed(TRAIN_SEED + 202)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    return train_loader, val_loader

def lr_multiplier(epoch):
    if epoch < 1: return epoch + 1
    progress = (epoch - 1) / max(1, EPOCHS - 1)
    return (1e-5 / LR) + (1 - 1e-5 / LR) * 0.5 * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def validate(model, loader, ssim_weight):
    model.eval(); loss_total = 0.0; count = 0
    cc_plane = torch.zeros(4, device=device); ss_plane = torch.zeros(4, device=device)
    source_sums = {0: {'cc': 0.0, 'ssim': 0.0, 'n': 0}, 1: {'cc': 0.0, 'ssim': 0.0, 'n': 0}}
    for target, sources in loader:
        target = target.to(device, non_blocking=True); sources = sources.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda'):
            phase = math.pi * torch.tanh(model(2 * target - 1))
        loss, cc, sv = doe_loss(target, physics(phase.float()), ssim_weight)
        loss_total += loss.item() * target.shape[0]; count += target.shape[0]
        cc_plane += cc.sum(0); ss_plane += sv.sum(0)
        for source_id in (0, 1):
            mask = sources == source_id; n = int(mask.sum().item())
            source_sums[source_id]['cc'] += float(cc[mask].sum().item())
            source_sums[source_id]['ssim'] += float(sv[mask].sum().item())
            source_sums[source_id]['n'] += n
    cc_values = (cc_plane / count).cpu().tolist(); ss_values = (ss_plane / count).cpu().tolist()
    row = {'val_loss': loss_total / count, 'val_cc': float(np.mean(cc_values)), 'val_ssim': float(np.mean(ss_values)),
           'val_cc_min': float(np.min(cc_values)), 'val_ssim_min': float(np.min(ss_values))}
    row.update({f'cc_z{m+1}': value for m, value in enumerate(cc_values)})
    row.update({f'ssim_z{m+1}': value for m, value in enumerate(ss_values)})
    for source_id, label in ((0, 'synthetic'), (1, 'bsds')):
        values = source_sums[source_id]; n = values['n']
        row[f'cc_{label}'] = values['cc'] / n; row[f'ssim_{label}'] = values['ssim'] / n
    return row

def run(name, architecture, ssim_weight):
    print(f'\n===== {name.upper()} =====', flush=True)
    set_seed(TRAIN_SEED); train_loader, val_loader = make_loaders()
    model = (BaselineUNet() if architecture == 'baseline' else SharedDepthUNet()).to(device)
    params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
    scaler = torch.amp.GradScaler('cuda'); history = []
    for epoch in range(EPOCHS):
        started = time.perf_counter(); model.train(); train_total = 0.0; seen = 0
        for batch_index, (target, _) in enumerate(train_loader, start=1):
            target = target.to(device, non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda'):
                phase = math.pi * torch.tanh(model(2 * target - 1))
            with torch.autocast(device_type='cuda', enabled=False):
                loss, _, _ = doe_loss(target, physics(phase.float()), ssim_weight)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); train_total += loss.item() * target.shape[0]; seen += target.shape[0]
            if batch_index == 1 or batch_index % LOG_EVERY == 0 or batch_index == len(train_loader):
                elapsed = time.perf_counter() - started; eta = elapsed / batch_index * (len(train_loader) - batch_index)
                print(f'{name} epoch {epoch+1}/{EPOCHS}: {batch_index}/{len(train_loader)}, loss={loss.item():.4f}, ETA={eta:.0f}s', flush=True)
        row = validate(model, val_loader, ssim_weight); scheduler.step()
        row.update({'epoch': epoch + 1, 'train_loss': train_total / seen,
                    'epoch_seconds': time.perf_counter() - started, 'ssim_weight': ssim_weight})
        history.append(row); print(row, flush=True)
    with (OUTPUT_DIR / f'metrics_{name}.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=history[0].keys()); writer.writeheader(); writer.writerows(history)
    del model, optimizer, scheduler, scaler, train_loader, val_loader; gc.collect(); torch.cuda.empty_cache()
    return {'history': history, 'parameter_count': params, 'ssim_weight': ssim_weight}

specs = [
    ('baseline_a11', 'baseline', 1.1),
    ('shared_a11', 'shared', 1.1),
    ('shared_a15', 'shared', 1.5),
    ('shared_a20', 'shared', 2.0),
]
results = {name: run(name, architecture, weight) for name, architecture, weight in specs}

METRICS = ('val_cc', 'val_ssim', 'val_cc_min', 'val_ssim_min', 'cc_synthetic', 'ssim_synthetic', 'cc_bsds', 'ssim_bsds', 'epoch_seconds')
def tail(name, metric): return float(np.mean([row[metric] for row in results[name]['history'][-3:]]))
summary = {name: {metric: tail(name, metric) for metric in METRICS} for name in results}
for name in results:
    summary[name]['parameter_count'] = results[name]['parameter_count']; summary[name]['ssim_weight'] = results[name]['ssim_weight']

baseline = summary['baseline_a11']; comparisons = {}; accepted = []
for name in ('shared_a11', 'shared_a15', 'shared_a20'):
    delta = {metric: summary[name][metric] - baseline[metric] for metric in METRICS if metric != 'epoch_seconds'}
    passed = (delta['val_cc'] >= .01 and delta['val_ssim'] >= -.003 and
              delta['val_cc_min'] >= 0 and delta['val_ssim_min'] >= -.003)
    comparisons[name] = {'delta': delta, 'passed': passed}
    if passed: accepted.append(name)
if accepted:
    selected = max(accepted, key=lambda name: (summary[name]['val_ssim'], summary[name]['val_cc']))
    decision = f'FULL_TRAIN_{selected.upper()}'
else:
    selected = None; decision = 'NO_WEIGHT_PASSED_TRY_MEAN_LOSS'
output = {'summary_tail3': summary, 'comparisons_to_baseline': comparisons, 'selected': selected,
          'decision': decision, 'config': {'data_seed': DATA_SEED, 'train_seed': TRAIN_SEED,
          'train_samples': TRAIN_SAMPLES, 'val_samples': VAL_SAMPLES, 'epochs': EPOCHS}}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(output, indent=2), encoding='utf-8')
print(json.dumps(output, indent=2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'baseline_a11': 'black', 'shared_a11': 'tab:blue', 'shared_a15': 'tab:orange', 'shared_a20': 'tab:green'}
for name in results:
    epochs = [row['epoch'] for row in results[name]['history']]
    axes[0].plot(epochs, [row['val_loss'] for row in results[name]['history']], marker='o', label=name, color=colors[name])
    axes[1].plot(epochs, [row['val_cc'] for row in results[name]['history']], marker='o', label=name, color=colors[name])
    axes[2].plot(epochs, [row['val_ssim'] for row in results[name]['history']], marker='o', label=name, color=colors[name])
for axis, title in zip(axes, ['Validation loss', 'Validation CC', 'Validation SSIM']):
    axis.set_title(title); axis.set_xlabel('epoch'); axis.grid(alpha=.3); axis.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'comparison.png', dpi=160); plt.show()
print('Saved to:', OUTPUT_DIR)